In [2]:
# RAPORT JAKOŚCI - WCZYTANIE I PODGLĄD DANYCH

import pandas as pd
from google.colab import files

df = pd.read_csv("fuel_prices.csv", sep=";")

display(df.head())

print("Wymiary danych:", df.shape)

,productName,effectiveDate,publishFrom,value,locationName,locationSymbol,unit,productId
0,Pb95,2026-11-11T00:00:00,2026-11-11T00:00:00,5307.0,NaN,NaN,NaN,41
1,Pb95,2026-11-10T00:00:00,2026-11-10T00:00:00,5346.0,NaN,NaN,NaN,41
2,Pb95,2026-11-07T00:00:00,2026-11-07T00:00:00,5296.0,NaN,NaN,NaN,41
3,Pb95,2026-11-06T00:00:00,2026-11-06T00:00:00,5368.0,NaN,NaN,NaN,41
4,Pb95,2026-11-05T00:00:00,2026-11-05T00:00:00,5438.0,NaN,NaN,NaN,41


Wymiary danych: (9715, 8)


In [3]:
# RAPORT JAKOŚCI DANYCH - BRAKI I DUPLIKATY

print("=== PODSTAWOWE INFORMACJE ===")
print("Liczba wierszy:", len(df))
print("Liczba kolumn:", len(df.columns))

print("\n=== TYPY DANYCH ===")
print(df.dtypes)

print("\n=== BRAKUJĄCE WARTOŚCI ===")
braki = pd.DataFrame({
    "Liczba braków": df.isna().sum(),
    "Procent braków (%)": (df.isna().mean() * 100).round(2)
})
display(braki)

print("\n=== DUPLIKATY ===")
print("Identyczne wiersze:", df.duplicated().sum())

print("\n=== DUPLIKATY PALIWO + DATA ===")
print(
    df.duplicated(
        subset=["productName", "effectiveDate"]
    ).sum()
)

=== PODSTAWOWE INFORMACJE ===
Liczba wierszy: 9715
Liczba kolumn: 8

=== TYPY DANYCH ===
productName        object
effectiveDate      object
publishFrom        object
value             float64
locationName      float64
locationSymbol    float64
unit               object
productId           int64
dtype: object

=== BRAKUJĄCE WARTOŚCI ===


,Liczba braków,Procent braków (%)
productName,0,0.00
effectiveDate,0,0.00
publishFrom,4080,42.00
value,0,0.00
locationName,9715,100.00
locationSymbol,9715,100.00
unit,7139,73.48
productId,0,0.00



=== DUPLIKATY ===
Identyczne wiersze: 0

=== DUPLIKATY PALIWO + DATA ===
0


In [4]:
# RAPORT JAKOŚCI - POPRAWNOŚĆ DAT I CEN

daty = pd.to_datetime(df["effectiveDate"], errors="coerce")
ceny = pd.to_numeric(df["value"], errors="coerce")

print("=== KONTROLA DAT ===")
print("Nieprawidłowe daty:", daty.isna().sum())
print("Najstarsza data:", daty.min())
print("Najnowsza data:", daty.max())

print("\n=== KONTROLA CEN ===")
print("Nieprawidłowe ceny:", ceny.isna().sum())
print("Ceny zerowe:", (ceny == 0).sum())
print("Ceny ujemne:", (ceny < 0).sum())
print("Najniższa cena:", ceny.min())
print("Najwyższa cena:", ceny.max())

print("\n=== RODZAJE PALIW ===")
display(df["productName"].value_counts())

print("\n=== STATYSTYKI CEN WEDŁUG PALIWA ===")
display(
    df.groupby("productName")["value"]
      .agg(["count", "min", "median", "max"])
      .round(2)
)

=== KONTROLA DAT ===
Nieprawidłowe daty: 0
Najstarsza data: 2019-01-01 00:00:00
Najnowsza data: 2026-11-11 00:00:00

=== KONTROLA CEN ===
Nieprawidłowe ceny: 0
Ceny zerowe: 0
Ceny ujemne: 0
Najniższa cena: 1530.0
Najwyższa cena: 8311.0

=== RODZAJE PALIW ===


,count
productName,
Pb95,1943
Pb98,1943
ONEkodiesel,1943
ONArctic2,1943
OnEkoterm,1943



=== STATYSTYKI CEN WEDŁUG PALIWA ===


,count,min,median,max
productName,,,,
ONArctic2,1943,3121.0,5114.0,8038.0
ONEkodiesel,1943,2876.0,4769.0,7704.0
OnEkoterm,1943,1530.0,3444.0,6372.0
Pb95,1943,2705.0,4607.0,7519.0
Pb98,1943,2958.0,4994.0,8311.0


In [5]:
# RAPORT JAKOŚCI - WARTOŚCI ODSTAJĄCE (IQR)

wyniki = []

for paliwo, grupa in df.groupby("productName"):
    q1 = grupa["value"].quantile(0.25)
    q3 = grupa["value"].quantile(0.75)
    iqr = q3 - q1

    dolna = q1 - 1.5 * iqr
    gorna = q3 + 1.5 * iqr

    odstajace = grupa[
        (grupa["value"] < dolna) |
        (grupa["value"] > gorna)
    ]

    wyniki.append({
        "Paliwo": paliwo,
        "Liczba wartości odstających": len(odstajace),
        "Procent (%)": round(
            len(odstajace) / len(grupa) * 100, 2
        ),
        "Dolna granica IQR": round(dolna, 2),
        "Górna granica IQR": round(gorna, 2)
    })

raport_iqr = pd.DataFrame(wyniki)
display(raport_iqr)

,Paliwo,Liczba wartości odstających,Procent (%),Dolna granica IQR,Górna granica IQR
0,ONArctic2,47,2.42,2312.25,7634.25
1,ONEkodiesel,82,4.22,2236.25,7102.25
2,OnEkoterm,0,0.00,85.00,7005.00
3,Pb95,25,1.29,2230.00,7022.00
4,Pb98,20,1.03,2079.50,7791.50


In [6]:
# RAPORT JAKOŚCI - LUKI CZASOWE

df_czas = df.copy()

df_czas["effectiveDate"] = pd.to_datetime(
    df_czas["effectiveDate"], errors="coerce"
)

wyniki_luk = []

for paliwo, grupa in df_czas.groupby("productName"):
    grupa = grupa.sort_values("effectiveDate")

    roznice = grupa["effectiveDate"].diff().dt.days

    wyniki_luk.append({
        "Paliwo": paliwo,
        "Liczba obserwacji": len(grupa),
        "Liczba przerw > 1 dzień": int((roznice > 1).sum()),
        "Najdłuższa przerwa (dni)": roznice.max(),
        "Mediana odstępu (dni)": roznice.median()
    })

raport_luki = pd.DataFrame(wyniki_luk)
display(raport_luki)

,Paliwo,Liczba obserwacji,Liczba przerw > 1 dzień,Najdłuższa przerwa (dni),Mediana odstępu (dni)
0,ONArctic2,1943,437,6.0,1.0
1,ONEkodiesel,1943,437,6.0,1.0
2,OnEkoterm,1943,437,6.0,1.0
3,Pb95,1943,437,6.0,1.0
4,Pb98,1943,437,6.0,1.0


In [7]:
# RAPORT JAKOŚCI - NAJDŁUŻSZE PRZERWY

pb95 = df_czas[
    df_czas["productName"] == "Pb95"
].sort_values("effectiveDate").copy()

pb95["Poprzednia data"] = pb95["effectiveDate"].shift(1)

pb95["Przerwa (dni)"] = (
    pb95["effectiveDate"] - pb95["Poprzednia data"]
).dt.days

najdluzsze = pb95[
    pb95["Przerwa (dni)"] >= 5
][["Poprzednia data", "effectiveDate", "Przerwa (dni)"]]

display(najdluzsze)

,Poprzednia data,effectiveDate,Przerwa (dni)
1864,2019-04-19,2019-04-24,5.0
1857,2019-05-03,2019-05-08,5.0
1625,2020-04-10,2020-04-15,5.0
1449,2020-12-24,2020-12-30,6.0
1383,2021-04-02,2021-04-07,5.0
1343,2021-06-03,2021-06-08,5.0
1201,2021-12-24,2021-12-30,6.0
1126,2022-04-15,2022-04-20,5.0
1117,2022-04-30,2022-05-05,5.0
1096,2022-06-02,2022-06-07,5.0


In [8]:
# RAPORT JAKOŚCI - SPRAWDZENIE JEDNOSTEK

print(df["unit"].value_counts(dropna=False))

unit
NaN       7139
PLN/m3    2576
Name: count, dtype: int64


In [9]:
# RAPORT JAKOŚCI - DATY Z PRZYSZŁOŚCI

print("Daty z przyszłości:")
print((daty > pd.Timestamp.today()).sum())

Daty z przyszłości:
0


In [10]:
# RAPORT JAKOŚCI - SZCZEGÓŁOWA KONTROLA WARTOŚCI ODSTAJĄCYCH

wyniki = []

for paliwo, grupa in df.groupby("productName"):
    q1 = grupa["value"].quantile(0.25)
    q3 = grupa["value"].quantile(0.75)
    iqr = q3 - q1

    dolna = q1 - 1.5 * iqr
    gorna = q3 + 1.5 * iqr

    odstajace = grupa[
        (grupa["value"] < dolna) |
        (grupa["value"] > gorna)
    ].copy()

    odstajace["Granica dolna"] = dolna
    odstajace["Granica gorna"] = gorna

    wyniki.append(odstajace)

wszystkie_odstajace = pd.concat(wyniki)

print("Liczba wartości odstających:", len(wszystkie_odstajace))

display(
    wszystkie_odstajace[
        ["productName", "effectiveDate", "value",
         "Granica dolna", "Granica gorna"]
    ].sort_values(["productName", "effectiveDate"])
)


Liczba wartości odstających: 174


,productName,effectiveDate,value,Granica dolna,Granica gorna
6983,ONArctic2,2022-03-09T00:00:00,8029.0,2312.25,7634.25
6982,ONArctic2,2022-03-10T00:00:00,7673.0,2312.25,7634.25
6918,ONArctic2,2022-06-16T00:00:00,7683.0,2312.25,7634.25
6917,ONArctic2,2022-06-18T00:00:00,7699.0,2312.25,7634.25
6916,ONArctic2,2022-06-21T00:00:00,7703.0,2312.25,7634.25
...,...,...,...,...,...
3029,Pb98,2022-06-22T00:00:00,7955.0,2079.50,7791.50
3028,Pb98,2022-06-23T00:00:00,7887.0,2079.50,7791.50
3027,Pb98,2022-06-24T00:00:00,7888.0,2079.50,7791.50
3026,Pb98,2022-06-25T00:00:00,7887.0,2079.50,7791.50


In [11]:
# RAPORT JAKOŚCI - WARTOŚCI ODSTAJĄCE WEDŁUG LAT

wszystkie_odstajace["rok"] = pd.to_datetime(
    wszystkie_odstajace["effectiveDate"]
).dt.year

print("=== WARTOŚCI ODSTAJĄCE WEDŁUG LAT ===")
display(
    pd.crosstab(
        wszystkie_odstajace["rok"],
        wszystkie_odstajace["productName"]
    )
)

print("\n=== LICZBA WARTOŚCI ODSTAJĄCYCH W KAŻDYM ROKU ===")
display(
    wszystkie_odstajace["rok"].value_counts().sort_index()
)


=== WARTOŚCI ODSTAJĄCE WEDŁUG LAT ===


productName,ONArctic2,ONEkodiesel,Pb95,Pb98
rok,,,,
2022,22,51,25,20
2026,25,31,0,0



=== LICZBA WARTOŚCI ODSTAJĄCYCH W KAŻDYM ROKU ===


,count
rok,
2022,118
2026,56


In [12]:
# RAPORT JAKOŚCI - SPRAWDZENIE WARTOŚCI ODSTAJĄCYCH

for paliwo in ["Pb95", "Pb98", "ONEkodiesel", "ONArctic2"]:
    grupa = df[df["productName"] == paliwo].copy()
    grupa["effectiveDate"] = pd.to_datetime(grupa["effectiveDate"])
    grupa = grupa.sort_values("effectiveDate")

    q1 = grupa["value"].quantile(0.25)
    q3 = grupa["value"].quantile(0.75)
    iqr = q3 - q1

    dolna = q1 - 1.5 * iqr
    gorna = q3 + 1.5 * iqr

    grupa["Odstajaca"] = (
        (grupa["value"] < dolna) |
        (grupa["value"] > gorna)
    )

    grupa["Poprzednia cena"] = grupa["value"].shift(1)
    grupa["Nastepna cena"] = grupa["value"].shift(-1)

    wynik = grupa[grupa["Odstajaca"]]

    print(f"\n=== {paliwo} ===")
    print("Liczba wartości odstających:", len(wynik))

    display(
        wynik[
            ["effectiveDate", "Poprzednia cena",
             "value", "Nastepna cena"]
        ].head(15)
    )



=== Pb95 ===
Liczba wartości odstających: 25


,effectiveDate,Poprzednia cena,value,Nastepna cena
1101,2022-05-27,6895.0,7076.0,7129.0
1100,2022-05-28,7076.0,7129.0,7213.0
1099,2022-05-31,7129.0,7213.0,7406.0
1098,2022-06-01,7213.0,7406.0,7519.0
1097,2022-06-02,7406.0,7519.0,7463.0
1096,2022-06-07,7519.0,7463.0,7397.0
1095,2022-06-08,7463.0,7397.0,7316.0
1094,2022-06-09,7397.0,7316.0,7323.0
1093,2022-06-10,7316.0,7323.0,7328.0
1092,2022-06-11,7323.0,7328.0,7323.0



=== Pb98 ===
Liczba wartości odstających: 20


,effectiveDate,Poprzednia cena,value,Nastepna cena
3044,2022-05-27,7504.0,7798.0,7864.0
3043,2022-05-28,7798.0,7864.0,7950.0
3042,2022-05-31,7864.0,7950.0,8194.0
3041,2022-06-01,7950.0,8194.0,8311.0
3040,2022-06-02,8194.0,8311.0,8232.0
3039,2022-06-07,8311.0,8232.0,8153.0
3038,2022-06-08,8232.0,8153.0,7997.0
3037,2022-06-09,8153.0,7997.0,7942.0
3036,2022-06-10,7997.0,7942.0,8005.0
3035,2022-06-11,7942.0,8005.0,8017.0



=== ONEkodiesel ===
Liczba wartości odstających: 82


,effectiveDate,Poprzednia cena,value,Nastepna cena
5041,2022-03-08,6632.0,7276.0,7704.0
5040,2022-03-09,7276.0,7704.0,7348.0
5039,2022-03-10,7704.0,7348.0,7060.0
5029,2022-03-24,6847.0,7198.0,7254.0
5028,2022-03-25,7198.0,7254.0,7187.0
5027,2022-03-26,7254.0,7187.0,7091.0
4983,2022-06-02,6912.0,7172.0,7166.0
4982,2022-06-07,7172.0,7166.0,7130.0
4981,2022-06-08,7166.0,7130.0,7254.0
4980,2022-06-09,7130.0,7254.0,7098.0



=== ONArctic2 ===
Liczba wartości odstających: 47


,effectiveDate,Poprzednia cena,value,Nastepna cena
6983,2022-03-09,7601.0,8029.0,7673.0
6982,2022-03-10,8029.0,7673.0,7385.0
6918,2022-06-16,7586.0,7683.0,7699.0
6917,2022-06-18,7683.0,7699.0,7703.0
6916,2022-06-21,7699.0,7703.0,7626.0
6841,2022-10-07,7605.0,7706.0,7997.0
6840,2022-10-08,7706.0,7997.0,7994.0
6839,2022-10-11,7997.0,7994.0,7873.0
6838,2022-10-12,7994.0,7873.0,7861.0
6837,2022-10-13,7873.0,7861.0,7882.0


In [13]:
# RAPORT JAKOŚCI - ANALIZA PRZERW CZASOWYCH

df_przerwy = df.copy()
df_przerwy["effectiveDate"] = pd.to_datetime(df_przerwy["effectiveDate"])

pb95 = df_przerwy[
    df_przerwy["productName"] == "Pb95"
].sort_values("effectiveDate").copy()

pb95["Poprzednia data"] = pb95["effectiveDate"].shift(1)

pb95["Przerwa (dni)"] = (
    pb95["effectiveDate"] - pb95["Poprzednia data"]
).dt.days

przerwy = pb95[pb95["Przerwa (dni)"] > 1].copy()

przerwy["Dzien poprzedniego notowania"] = (
    przerwy["Poprzednia data"].dt.day_name()
)

przerwy["Dzien kolejnego notowania"] = (
    przerwy["effectiveDate"].dt.day_name()
)

przerwy["Dni robocze pomiedzy"] = [
    len(pd.bdate_range(start, end)) - 2
    for start, end in zip(
        przerwy["Poprzednia data"],
        przerwy["effectiveDate"]
    )
]

print("=== ANALIZA PRZERW CZASOWYCH PB95 ===")
print("Liczba przerw > 1 dzień:", len(przerwy))

print("Przerwy bez pominiętych dni roboczych:",
      (przerwy["Dni robocze pomiedzy"] == 0).sum())

print("Przerwy z pominiętymi dniami roboczymi:",
      (przerwy["Dni robocze pomiedzy"] > 0).sum())

print("\n=== ROZKŁAD DŁUGOŚCI PRZERW ===")
display(przerwy["Przerwa (dni)"].value_counts().sort_index())

print("\n=== NAJDŁUŻSZE PRZERWY ===")
display(
    przerwy[
        ["Poprzednia data", "effectiveDate",
         "Przerwa (dni)", "Dni robocze pomiedzy"]
    ].sort_values("Przerwa (dni)", ascending=False).head(20)
)


=== ANALIZA PRZERW CZASOWYCH PB95 ===
Liczba przerw > 1 dzień: 437
Przerwy bez pominiętych dni roboczych: 361
Przerwy z pominiętymi dniami roboczymi: 73

=== ROZKŁAD DŁUGOŚCI PRZERW ===


,count
Przerwa (dni),
2.0,28
3.0,351
4.0,37
5.0,17
6.0,4



=== NAJDŁUŻSZE PRZERWY ===


,Poprzednia data,effectiveDate,Przerwa (dni),Dni robocze pomiedzy
1449,2020-12-24,2020-12-30,6.0,3
377,2025-05-01,2025-05-07,6.0,3
217,2025-12-24,2025-12-30,6.0,3
1201,2021-12-24,2021-12-30,6.0,3
711,2023-12-23,2023-12-28,5.0,2
1625,2020-04-10,2020-04-15,5.0,2
622,2024-05-03,2024-05-08,5.0,2
384,2025-04-18,2025-04-23,5.0,2
1343,2021-06-03,2021-06-08,5.0,2
644,2024-03-29,2024-04-03,5.0,2


In [16]:
# RAPORT JAKOŚCI - KONTROLA KOLUMNY publishFrom

print("=== BRAKI W publishFrom ===")
print("Liczba braków:", df["publishFrom"].isna().sum())
print("Procent braków:",
      round(df["publishFrom"].isna().mean() * 100, 2), "%")

print("\n=== BRAKI WEDŁUG PALIWA ===")
display(
    df.groupby("productName")["publishFrom"]
    .apply(lambda x: pd.Series({
        "Liczba braków": x.isna().sum(),
        "Procent braków": round(x.isna().mean() * 100, 2)
    }))
    .unstack()
)

print("\n=== PORÓWNANIE DAT ===")
porownanie = df[
    df["publishFrom"].notna()
].copy()

porownanie["effectiveDate"] = pd.to_datetime(
    porownanie["effectiveDate"], errors="coerce"
)
porownanie["publishFrom"] = pd.to_datetime(
    porownanie["publishFrom"], errors="coerce"
)

print("Daty identyczne:",
      (porownanie["effectiveDate"] == porownanie["publishFrom"]).sum())

print("Daty różne:",
      (porownanie["effectiveDate"] != porownanie["publishFrom"]).sum())

display(
    porownanie[
        porownanie["effectiveDate"] != porownanie["publishFrom"]
    ][["productName", "effectiveDate", "publishFrom"]].head(10)
)


=== BRAKI W publishFrom ===
Liczba braków: 4080
Procent braków: 42.0 %

=== BRAKI WEDŁUG PALIWA ===


,Liczba braków,Procent braków
productName,,
ONArctic2,816.0,42.0
ONEkodiesel,816.0,42.0
OnEkoterm,816.0,42.0
Pb95,816.0,42.0
Pb98,816.0,42.0



=== PORÓWNANIE DAT ===
Daty identyczne: 5635
Daty różne: 0


,productName,effectiveDate,publishFrom
